In [5]:
"""
Script Function: Extract Hospital 3 Contract Rules via Gemini API
Reads the base agreement, rate schedule, and amendment, combines them,
prompts the LLM to apply the amendment and extract final rules, and saves to JSON.
"""
import os
import json
import google.generativeai as genai
from dotenv import load_dotenv

# 1. Load environment variables
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)

# 2. Initialize the Gemini model
model = genai.GenerativeModel('gemini-3.1-flash-lite')

# 3. Read all relevant contract parts for Hospital 3
h3_dir = '../data/contracts/hospital_3/'

with open(os.path.join(h3_dir, 'base_agreement.md'), 'r', encoding='utf-8') as f:
    base_agreement = f.read()
    
with open(os.path.join(h3_dir, 'appendix_b_rate_schedule.md'), 'r', encoding='utf-8') as f:
    appendix = f.read()
    
with open(os.path.join(h3_dir, 'amendment_no_1.md'), 'r', encoding='utf-8') as f:
    amendment = f.read()

# Combine them into one text for the LLM
combined_contract = f"--- BASE AGREEMENT ---\n{base_agreement}\n\n--- APPENDIX B (RATES) ---\n{appendix}\n\n--- AMENDMENT NO 1 ---\n{amendment}"

# 4. Define the Prompt (Focusing on applying amendments)
prompt = """
You are an expert medical billing auditor. I will provide you with a hospital provider services agreement that is split into a base agreement, a rate schedule, and an amendment.

CRITICAL INSTRUCTION: You must apply any changes, price updates, or new rules mentioned in "AMENDMENT NO 1" over the original base agreement or appendix.

Extract the following details into a structured JSON format:
1. contract_metadata: provider_name, contract_number, effective_dates (start and end), currency.
2. services: a list of services where each service has:
   - service_name: The official name of the service.
   - unit_basis: The basis for billing.
   - unit_price_cents: The FINAL price in cents (integer) AFTER applying the amendment. If the price is in major currency, convert to cents.
   - conditions: Any special conditions, limits, thresholds, or discounts. Update these if the amendment changes them (leave empty string if none).

Return ONLY a valid JSON object. Do not include markdown blocks like ```json ... ```.

Here is the combined contract:
"""

# 5. Save the prompt for documentation
os.makedirs('../prompts', exist_ok=True)
with open('../prompts/prompt_h3_v1.txt', 'w', encoding='utf-8') as f:
    f.write(prompt)

print("⏳ Sending combined Hospital 3 contract to Gemini API... Please wait.")

# 6. Call the API
response = model.generate_content(prompt + "\n\n" + combined_contract)

# 7. Clean and parse the response
extracted_text = response.text.strip()
if extracted_text.startswith("```json"):
    extracted_text = extracted_text[7:-3].strip()
elif extracted_text.startswith("```"):
    extracted_text = extracted_text[3:-3].strip()

# 8. Save the parsed JSON
try:
    contract_rules = json.loads(extracted_text)
    
    output_path = '../data/contracts/hospital_3/contract_rules.json'
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(contract_rules, f, indent=4)
        
    print(f"✅ Contract rules successfully extracted and saved to {output_path}")
    print("\nPreview of the extracted services:")
    for service in contract_rules.get("services", [])[:3]:
        print(f"- {service['service_name']}: {service['unit_price_cents']} cents ({service['unit_basis']})")

except json.JSONDecodeError:
    print("❌ Failed to parse JSON. Raw output:")
    print(extracted_text)

⏳ Sending combined Hospital 3 contract to Gemini API... Please wait.
✅ Contract rules successfully extracted and saved to ../data/contracts/hospital_3/contract_rules.json

Preview of the extracted services:
- Advanced Gastrointestinal Telemetry Monitoring: 128100 cents (per day of service)
- Advanced Geriatric Nutritional Support: 1250 cents (per unit dispensed)
- Advanced Metabolic Discharge Planning: 21075 cents (per hour)


In [2]:
"""
Script Function: Item Mapping (Fuzzy Matching) for Hospital 3
Matches the raw billing descriptions from Hospital 3 invoices to the official contract terms.
Saves the output to 'service_mapping.json'.
"""

import pandas as pd
import json
from rapidfuzz import process, fuzz

# 1. Setup relative paths for Hospital 3
line_items_path = '../data/invoices/hospital_3_line_items.csv'
contract_rules_path = '../data/contracts/hospital_3/contract_rules.json'
mapping_output_path = '../data/contracts/hospital_3/service_mapping.json'

# 2. Load data
line_items_df = pd.read_csv(line_items_path)
with open(contract_rules_path, 'r', encoding='utf-8') as f:
    contract_rules = json.load(f)

# 3. Extract official service names and billed descriptions
official_services = [service['service_name'] for service in contract_rules['services']]
billed_descriptions = line_items_df['description'].unique()

print("⏳ Mapping billed descriptions to official contract services for Hospital 3...")

# 4. Perform Fuzzy Matching
mapping_dict = {}
for billed_desc in billed_descriptions:
    # Clean description (remove specific codes if present)
    clean_desc = billed_desc.split('/')[0].strip() if '/' in billed_desc else billed_desc
    
    # Find the best match using RapidFuzz
    best_match = process.extractOne(clean_desc, official_services, scorer=fuzz.token_sort_ratio)
    
    if best_match:
        matched_service, score, _ = best_match
        mapping_dict[billed_desc] = {
            "mapped_service": matched_service,
            "confidence_score": round(score, 2)
        }

# 5. Save the mapping to a JSON file
with open(mapping_output_path, 'w', encoding='utf-8') as f:
    json.dump(mapping_dict, f, indent=4)

print(f"✅ Mapping complete for Hospital 3! Saved to: {mapping_output_path}")

⏳ Mapping billed descriptions to official contract services for Hospital 3...
✅ Mapping complete for Hospital 3! Saved to: ../data/contracts/hospital_3/service_mapping.json


In [1]:
"""
Script Function: Invoice Auditor Engine & Submission Generator (Hospital 3)
Audits hospital 3 invoices using the correct directory paths and safely appends results to submission.csv.
"""
import pandas as pd
import json
import os

# 1. Setup correct paths based on your project directory structure
invoices_path = '../data/invoices/hospital_3_invoices.csv'
line_items_path = '../data/invoices/hospital_3_line_items.csv'
contract_rules_path = '../data/contracts/hospital_3/contract_rules.json'
mapping_path = '../data/contracts/hospital_3/service_mapping.json'

# 2. Load data
invoices_df = pd.read_csv(invoices_path)
line_items_df = pd.read_csv(line_items_path)

with open(contract_rules_path, 'r', encoding='utf-8') as f:
    contract_rules = json.load(f)
    
with open(mapping_path, 'r', encoding='utf-8') as f:
    service_mapping = json.load(f)

rules_dict = {item['service_name']: item for item in contract_rules['services']}

# 3. Compute expected totals and apply confidence threshold
print("⏳ Auditing Hospital 3 line items...")
expected_totals = []
error_cats = []
confidences = []

for idx, row in line_items_df.iterrows():
    billed_desc = row['description']
    quantity = row['quantity']
    
    mapping_data = service_mapping.get(billed_desc, {})
    official_service = mapping_data.get('mapped_service')
    confidence = mapping_data.get('confidence_score', 0)
    
    # Check confidence threshold (>= 60)
    if official_service and official_service in rules_dict and confidence >= 60:
        base_price = rules_dict[official_service]['unit_price_cents']
        expected_total = quantity * base_price
        error_cat = "Clean"
    else:
        expected_total = 0
        error_cat = "Unmapped Service"
        confidence = 0.0
        
    expected_totals.append(expected_total)
    error_cats.append(error_cat)
    confidences.append(confidence / 100.0)

line_items_df['expected_line_total'] = expected_totals
line_items_df['line_confidence'] = confidences

# 4. Aggregate to invoice level
print("⏳ Aggregating to invoice level...")
invoice_summary = line_items_df.groupby('invoice_id').agg(
    expected_total_cents=('expected_line_total', 'sum'),
    min_confidence=('line_confidence', 'min')
).reset_index()

final_audit_df = invoices_df.merge(invoice_summary, on='invoice_id', how='left')

# Handle NaNs and apply tolerance for flagging
final_audit_df['expected_total_cents'] = final_audit_df['expected_total_cents'].fillna(0)
final_audit_df['min_confidence'] = final_audit_df['min_confidence'].fillna(0.0)
final_audit_df['flagged'] = (abs(final_audit_df['invoice_total_cents'] - final_audit_df['expected_total_cents']) > 2).astype(int)

def categorize_error(row):
    if row['flagged'] == 0:
        return ""
    if row['min_confidence'] < 0.6:
        return "Low Confidence Mapping"
    if row['invoice_total_cents'] > row['expected_total_cents']:
        return "Overbilled"
    return "Underbilled"

final_audit_df['error_category'] = final_audit_df.apply(categorize_error, axis=1)

# 5. Format and Safely Append to submission.csv
print("⏳ Appending Hospital 3 results to submission.csv...")
new_submission_df = pd.DataFrame({
    'invoice_id': final_audit_df['invoice_id'],
    'flagged': final_audit_df['flagged'],
    'error_category': final_audit_df['error_category'],
    'expected_total_cents': final_audit_df['expected_total_cents'].astype(int),
    'billed_total_cents': final_audit_df['invoice_total_cents'].astype(int),
    'confidence': final_audit_df['min_confidence'].round(2)
})

submission_path = '../submission.csv'
new_submission_df['hospital_temp'] = 'hospital_3'

if os.path.exists(submission_path):
    existing_df = pd.read_csv(submission_path)
    if 'hospital_temp' not in existing_df.columns:
        existing_df['hospital_temp'] = 'hospital_2'
    combined_df = pd.concat([existing_df, new_submission_df]).drop_duplicates(subset=['hospital_temp', 'invoice_id'], keep='last')
else:
    combined_df = new_submission_df

combined_df.drop(columns=['hospital_temp']).to_csv(submission_path, index=False)

print("✅ تم تدقيق المستشفى 3 بنجاح، وإضافته للملف المشترك بدون مسح النتائج السابقة!")
print(f"📊 إجمالي الفواتير في ملف submission.csv الآن: {len(combined_df)}")

⏳ Auditing Hospital 3 line items...
⏳ Aggregating to invoice level...
⏳ Appending Hospital 3 results to submission.csv...
✅ تم تدقيق المستشفى 3 بنجاح، وإضافته للملف المشترك بدون مسح النتائج السابقة!
📊 إجمالي الفواتير في ملف submission.csv الآن: 2989


In [2]:
import pandas as pd
import os

# Read the final combined submission file
submission_path = '../submission.csv'

if os.path.exists(submission_path):
    submission_df = pd.read_csv(submission_path)
    
    print("📊 --- Pipeline Health & Diagnostic Metrics Summary ---")
    print(f"Total audited invoices in submission file: {len(submission_df)}")
    
    # Calculate flagged ratio
    flagged_count = submission_df['flagged'].sum()
    flagged_ratio = (flagged_count / len(submission_df)) * 100
    print(f"Flagged Invoices Count: {flagged_count} ({flagged_ratio:.2f}%)")
    
    print("\n📋 Error Categories Breakdown:")
    print(submission_df['error_category'].value_counts())
    
    print("\n📈 Confidence Score Summary:")
    print(submission_df['confidence'].describe())
else:
    print("❌ submission.csv file not found. Make sure to run the auditing scripts for the hospitals first.")

📊 --- Pipeline Health & Diagnostic Metrics Summary ---
Total audited invoices in submission file: 2989
Flagged Invoices Count: 2970 (99.36%)

📋 Error Categories Breakdown:
error_category
Low Confidence Mapping    2954
Underbilled                 12
Overbilled                   4
Name: count, dtype: int64

📈 Confidence Score Summary:
count    2989.000000
mean        0.211542
std         0.178799
min         0.000000
25%         0.000000
50%         0.190000
75%         0.390000
max         0.670000
Name: confidence, dtype: float64
